# 07 — SHAP Token Attributions

Generates token-level SHAP explanations for the fine-tuned MentalBERT model.

**Run this on the training machine** — needs checkpoints in `results/runs/mentalbert/`.

Outputs:
- `results/figures/shap/` — one PNG per example (paper figures)
- Interactive HTML plots inline (for exploration)

Expected runtime: ~10–20 min for 32 examples on GPU.

In [ ]:
import sys
sys.path.insert(0, '..')

import shap
import torch
import numpy as np

from src.data import load_splits
from src.shap_explain import (
    load_model, get_predictions, select_examples,
    make_predict_fn, run_shap, save_figures, ID2LABEL,
)

print('CUDA available:', torch.cuda.is_available())

## Config
Change `RUN_DIR` and `MODEL_NAME` to switch to RoBERTa.

In [ ]:
RUN_DIR     = '../results/runs/mentalbert'
MODEL_NAME  = 'mental/mental-bert-base-uncased'
MAX_LENGTH  = 256
FIGURES_DIR = '../results/figures/shap'

N_CORRECT = 6  # high-confidence correct examples per class
N_WRONG   = 2  # borderline wrong examples per class

## 1 — Load model and data

In [ ]:
model, tokenizer, device = load_model(RUN_DIR, MODEL_NAME)
_, _, test_df = load_splits()
print(f'Test set: {len(test_df)} rows  |  device: {device}')

## 2 — Inference on full test set

In [ ]:
preds, probs = get_predictions(
    model, tokenizer, test_df['text'].tolist(), device, max_length=MAX_LENGTH,
)

accuracy = (preds == test_df['label_id'].values).mean()
print(f'Test accuracy: {accuracy:.4f}')

for cls, name in ID2LABEL.items():
    mask = test_df['label_id'].values == cls
    print(f'  {name:<12} recall={(preds[mask]==cls).mean():.3f}  n={mask.sum()}')

## 3 — Select curated examples

`N_CORRECT` high-confidence correct + `N_WRONG` borderline failures per class.
Failures are the most interesting for the paper.

In [ ]:
selected    = select_examples(test_df, preds, probs, n_correct=N_CORRECT, n_wrong=N_WRONG)
all_examples = [ex for cls_exs in selected.values() for ex in cls_exs]
all_texts    = [ex['text'] for ex in all_examples]

print(f'Total examples selected: {len(all_examples)}')
for cls, name in ID2LABEL.items():
    print(f'  {name}: {len(selected[cls])}')

## 4 — Run SHAP

Slow step (~10–20 min on GPU). A progress bar appears below.

In [ ]:
predict_fn  = make_predict_fn(model, tokenizer, device, max_length=MAX_LENGTH)
shap_values = run_shap(all_texts, predict_fn, tokenizer)

## 5 — Save paper figures

One PNG per example → `results/figures/shap/`

In [ ]:
save_figures(shap_values, all_examples, output_dir=FIGURES_DIR)

## 6 — Interactive HTML gallery

Red tokens push toward the predicted class; blue push away.

In [ ]:
for cls, name in ID2LABEL.items():
    idx = next(i for i, ex in enumerate(all_examples) if ex['true_label'] == name)
    ex  = all_examples[idx]
    print(f'\n── {name} | pred={ex["pred_label"]} conf={ex["confidence"]:.2f} ──')
    print(ex['text'][:200])
    shap.plots.text(shap_values[idx])

## 7 — Beeswarm (optional)

Overview of which tokens matter most across all examples for the severe class.

In [ ]:
# shap.plots.beeswarm(shap_values[:, :, 3])  # index 3 = severe
# NOTE: beeswarm expects uniform feature dimensions across examples; text SHAP
# values are variable-length per example so this plot is not compatible here.
# Use shap.plots.text() in step 6 above for token attribution visualisation.